# Entropy, cross-entropy & KL: hands-on (optional)
**LLMs, Animated · Lesson P1.6**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (or a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("ready")

## 1 · Surprise
The surprise of an outcome with probability $p$ is $-\log p$. With `base=2` it's in **bits**; with `base=np.e` in **nats**.

Implement `surprise(p, base=2)`. It should work on a single number *and* on a NumPy array. *(Hint: $\log_b x = \ln x / \ln b$.)*

In [ ]:
def surprise(p, base=2):
    # TODO: return -log_base(p)
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
assert np.isclose(surprise(0.5), 1.0)
assert np.isclose(surprise(1 / 8), 3.0)
assert np.isclose(surprise(1.0), 0.0)
assert np.isclose(surprise(0.25, base=np.e), np.log(4))          # 1.386 nats
assert np.allclose(surprise(np.array([0.5, 0.25])), [1, 2])
print("✓ exercise 1:  a 1-in-100 event is", round(float(surprise(0.01)), 2), "bits")

## 2 · Entropy = average surprise
$$H(p) = \sum_x p(x)\,\big(-\log p(x)\big)$$
Implement `entropy(p, base=2)` for a probability vector `p`. Outcomes with $p(x) = 0$ contribute **0** (the limit of $p \log p$ as $p \to 0$), so skip them rather than computing `log(0)`.

In [ ]:
def entropy(p, base=2):
    p = np.asarray(p, dtype=float)
    # TODO: sum of p * surprise(p) over the entries with p > 0
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
assert np.isclose(entropy([0.5, 0.5]), 1.0)                 # fair coin
assert np.isclose(entropy([0.9, 0.1]), 0.4690, atol=1e-4)    # biased coin
assert np.isclose(entropy(np.ones(6) / 6), np.log2(6))       # fair die
assert np.isclose(entropy([1, 0, 0, 0]), 0.0)                # one-hot: no uncertainty
assert np.isclose(entropy([0.5, 0.25, 0.125, 0.125]), 1.75)  # the lesson's animation
print("✓ exercise 2")

## 3 · Cross-entropy and KL
$$H(p, q) = \sum_x p(x)\,\big(-\log q(x)\big) \qquad D_{\mathrm{KL}}(p\,\|\,q) = H(p, q) - H(p) = \sum_x p(x) \log\frac{p(x)}{q(x)}$$
Implement both. Again skip terms where $p(x) = 0$. (If $q(x) = 0$ where $p(x) > 0$, the answer is infinite; NumPy will give `inf`, which is correct.)

In [ ]:
def cross_entropy(p, q, base=2):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    # TODO
    raise NotImplementedError("Exercise 3a")

def kl(p, q, base=2):
    # TODO: use the sum of p * log(p / q) form (not H(p,q) - H(p)), so the check below is meaningful
    raise NotImplementedError("Exercise 3b")

**Check 3**

In [ ]:
p = [0.5, 0.25, 0.125, 0.125]
q = [0.25, 0.25, 0.25, 0.25]
assert np.isclose(cross_entropy(p, q), 2.0)
assert np.isclose(kl(p, q), 0.25)
assert np.isclose(kl(p, q), cross_entropy(p, q) - entropy(p))   # the identity
assert np.isclose(kl(p, p), 0.0)
assert np.isclose(kl([0.9, 0.1], [0.5, 0.5]), 0.5310, atol=1e-4)
assert np.isclose(kl([0.5, 0.5], [0.9, 0.1]), 0.7370, atol=1e-4)
print("✓ exercise 3:  KL is not symmetric:", round(kl([0.9, 0.1], [0.5, 0.5]), 3), "vs", round(kl([0.5, 0.5], [0.9, 0.1]), 3), "bits")

## 4 · Test the two big claims on random distributions
The lesson claims $D_{\mathrm{KL}}(p\,\|\,q) \ge 0$ always, and that usually $D_{\mathrm{KL}}(p\,\|\,q) \ne D_{\mathrm{KL}}(q\,\|\,p)$. Don't trust it: test it.

Draw 5,000 random pairs of 5-outcome distributions (`rng.dirichlet(np.ones(5))` gives a random probability vector). Fill in:
- `min_kl`: the smallest $D_{\mathrm{KL}}(p\,\|\,q)$ you saw,
- `frac_asym`: the fraction of pairs where the two directions differ by more than 0.001 bits.

In [ ]:
rng = np.random.default_rng(0)
min_kl = None
frac_asym = None
# TODO: loop over 5000 random (p, q) pairs and fill in the two numbers above

**Check 4**

In [ ]:
assert min_kl is not None and frac_asym is not None
assert min_kl >= -1e-12, "KL should never be negative (tiny float error aside)"
assert frac_asym > 0.9, "for random pairs, the two directions almost always differ"
print(f"✓ exercise 4:  min KL = {min_kl:.5f} bits, asymmetric in {frac_asym:.1%} of pairs")

## 5 · The loss of a toy next-token predictor
A tiny "language model" outputs **logits** (raw scores) over a 5-token vocabulary at 4 positions. Softmax turns each row into probabilities (lesson P3.2):
$$q_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$
The training loss is the mean cross-entropy against the actual next tokens (one-hot targets), **in nats**: the average of $-\ln q(\text{actual token})$. Perplexity is $e^{\text{loss}}$.

Implement `log_softmax` (subtract the row max before exponentiating, for numerical stability) and `lm_loss`.

In [ ]:
vocab = ["the", "cat", "sat", "on", "mat"]
logits = np.array([
    [2.0, 0.5, 0.1, 0.3, 0.2],    # position 1: model expects "the"
    [0.2, 3.0, 0.1, 0.4, 1.0],    # position 2: "cat"
    [0.1, 0.3, 1.5, 0.2, 1.4],    # position 3: torn between "sat" and "mat"
    [0.5, 0.2, 0.1, 0.4, 0.3],    # position 4: no idea
])
targets = np.array([0, 1, 2, 3])  # the tokens that actually came next: the cat sat on


In [ ]:
def log_softmax(z):
    # TODO: return log of softmax along the last axis, computed stably
    raise NotImplementedError("Exercise 5a")

def lm_loss(logits, targets):
    # TODO: mean over positions of -log q(actual token), in nats
    raise NotImplementedError("Exercise 5b")

**Check 5**

In [ ]:
lp = log_softmax(logits)
assert np.allclose(np.exp(lp).sum(axis=1), 1.0), "each row of probabilities must sum to 1"
assert np.allclose(log_softmax(logits + 1000.0), lp), "adding a constant to every logit must not change anything (and must not overflow)"
loss = lm_loss(logits, targets)
per_token = -lp[np.arange(4), targets]
assert np.isclose(loss, per_token.mean())
assert np.isclose(loss, 0.8366, atol=1e-3), loss
print(f"✓ exercise 5:  per-token loss {np.round(per_token, 3)}  →  mean {loss:.3f} nats, perplexity {np.exp(loss):.2f}")
try:   # optional: compare with PyTorch's built-in (pre-installed on Colab)
    import torch, torch.nn.functional as F
    t = F.cross_entropy(torch.tensor(logits), torch.tensor(targets)).item()
    print("   PyTorch F.cross_entropy agrees:", round(t, 4))
    assert np.isclose(t, loss)
except ImportError:
    pass

## Experiment · the cross-entropy curve
Run this to see the picture from the lesson's playground: for a fixed reality `p_rain`, cross-entropy as a function of the forecast `q`. Its minimum sits exactly at $q = p$, at height $H(p)$, and the gap above that floor is the KL divergence. Change `p_rain` and re-run. Then try plotting `kl(q, p)` too, to see the asymmetry.

In [ ]:
p_rain = 0.3
qs = np.linspace(0.01, 0.99, 197)
ce = [cross_entropy([p_rain, 1 - p_rain], [q, 1 - q]) for q in qs]
plt.figure(figsize=(7, 4))
plt.plot(qs, ce, label="H(p, q)  (cross-entropy)")
plt.axhline(entropy([p_rain, 1 - p_rain]), ls="--", color="gray", label="H(p)  (entropy floor)")
plt.axvline(p_rain, color="tab:orange", lw=1, label="q = p")
plt.ylim(0, 4); plt.xlabel("forecast q(rain)"); plt.ylabel("bits"); plt.legend()
plt.title("Cross-entropy is lowest when the forecast matches reality")
plt.show()

## Recap
- Surprise $-\log p$; entropy $H(p)$ is its average under $p$.
- Cross-entropy $H(p, q)$ averages $q$'s surprise under the true $p$; with one-hot targets it's $-\log q(\text{actual})$, **the LLM loss** (in nats in PyTorch).
- $D_{\mathrm{KL}}(p\,\|\,q) = H(p, q) - H(p) \ge 0$, zero only when $q = p$, and not symmetric. You just verified both on 5,000 random pairs.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def surprise(p, base=2):
    return -np.log(p) / np.log(base)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def entropy(p, base=2):
    p = np.asarray(p, dtype=float)
    nz = p > 0
    return float(np.sum(p[nz] * surprise(p[nz], base)))

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def cross_entropy(p, q, base=2):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    nz = p > 0
    return float(np.sum(p[nz] * surprise(q[nz], base)))

def kl(p, q, base=2):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    nz = p > 0
    return float(np.sum(p[nz] * np.log(p[nz] / q[nz])) / np.log(base))

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
rng = np.random.default_rng(0)
kls, asym = [], 0
for _ in range(5000):
    p, q = rng.dirichlet(np.ones(5)), rng.dirichlet(np.ones(5))
    a, b = kl(p, q), kl(q, p)
    kls.append(a)
    asym += abs(a - b) > 1e-3
min_kl = min(kls)
frac_asym = asym / 5000

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def log_softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))

def lm_loss(logits, targets):
    lp = log_softmax(logits)
    return float(-lp[np.arange(len(targets)), targets].mean())